In [5]:
import pandas as pd
import os
import numpy as np
from sklearn.experimental import enable_iterative_imputer
import sklearn.impute as SKI
import sklearn.preprocessing as SKP
from sklearn.linear_model import BayesianRidge

# Define the input and output directories
input_directory = "/home/ahmedmas/Projects/Data_imputation/Data_After_neg"
output_directory_base = "/home/ahmedmas/Projects/Data_imputation/result_imputed_data"

# Function to drop unnecessary columns
def drop_unnecessary_columns(df):
    return df.drop(columns=['datetime', 'missing_info', 'removed_values'])

# Function to perform MICE Imputation
def MICE_imputation(input_data_pd):
    mice_imputer = SKI.IterativeImputer(
        estimator=BayesianRidge(),
        missing_values=np.nan,
        sample_posterior=False,
        max_iter=10,
        tol=0.001,
        n_nearest_features=None,
        initial_strategy="mean",
        imputation_order="ascending"
    )

    imputed_data_pd = pd.DataFrame(
        mice_imputer.fit_transform(input_data_pd),
        columns=input_data_pd.columns,
        index=input_data_pd.index)

    return imputed_data_pd

# Function to perform KNN Imputation
def KNN_imputation(input_data_pd):
    scaler = SKP.MinMaxScaler(feature_range=(0, 1))
    df_knn = pd.DataFrame(
        scaler.fit_transform(input_data_pd),
        columns=input_data_pd.columns,
        index=input_data_pd.index,
    )

    knn_imputer = SKI.KNNImputer(
        missing_values=np.nan,
        n_neighbors=5,
        weights='uniform',
        metric='nan_euclidean'
    )
    
    imputed_data_pd = pd.DataFrame(
        knn_imputer.fit_transform(df_knn),
        columns=df_knn.columns,
        index=df_knn.index,
    )

    # Rescale back to original range
    imputed_data_pd = pd.DataFrame(
        scaler.inverse_transform(imputed_data_pd),
        columns=imputed_data_pd.columns,
        index=imputed_data_pd.index,
    )

    return imputed_data_pd

# Function to process each file
def process_file(file_path, method):
    df = pd.read_csv(file_path)

    # Keep original PM2.5 column for later comparison
    df['PM2.5 (original)'] = df['PM2.5']

    # Drop unnecessary columns before imputation
    df_for_imputation = drop_unnecessary_columns(df).copy()

    # Apply the selected imputation method
    if method == "MICE":
        imputed_data_pd = MICE_imputation(df_for_imputation)
    elif method == "KNN":
        imputed_data_pd = KNN_imputation(df_for_imputation)

    # Get only imputed values for PM2.5
    df['PM2.5(only_imputed data)'] = imputed_data_pd['PM2.5']

    # Replace original non-missing values in 'PM2.5(only_imputed data)' with NaN
    df.loc[~df['PM2.5'].isna(), 'PM2.5(only_imputed data)'] = np.nan

    # Merge the original and imputed values
    df['full PM2.5 (after imputation)'] = df['PM2.5 (original)'].combine_first(df['PM2.5(only_imputed data)'])

    # Define output directory and create it if it doesn't exist
    output_directory = os.path.join(output_directory_base, method)
    os.makedirs(output_directory, exist_ok=True)

    # Save the final result with all necessary columns
    output_df = df[['datetime', 'missing_info', 'removed_values', 'PM2.5 (original)', 'PM2.5(only_imputed data)', 'full PM2.5 (after imputation)']]
    output_file_path = os.path.join(output_directory, os.path.basename(file_path))
    output_df.to_csv(output_file_path, index=False)

# Process all files in the input directory
for file_name in os.listdir(input_directory):
    if file_name.endswith('.csv'):
        file_path = os.path.join(input_directory, file_name)
        
        # Process with MICE Imputation
        process_file(file_path, method="MICE")
        
        # Process with KNN Imputation
        process_file(file_path, method="KNN")


In [6]:
import pandas as pd
import os
import numpy as np

# Define the input and output directories
input_directory = "/home/ahmedmas/Projects/Data_imputation/Data_After_neg"
output_directory_base = "/home/ahmedmas/Projects/Data_imputation/result_imputed_data"

# Function to drop unnecessary columns if they exist
def drop_unnecessary_columns(df):
    columns_to_drop = ['missing_info', 'removed_values']
    existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
    return df.drop(columns=existing_columns_to_drop)

# Function to perform Interpolation Imputation
def Interpolation_imputation(input_data_pd):
    imputed_data_pd = input_data_pd.interpolate(method='linear', limit_direction='both')
    return imputed_data_pd

# Function to perform Forward Fill Imputation
def ForwardFill_imputation(input_data_pd):
    imputed_data_pd = input_data_pd.fillna(method='ffill')
    return imputed_data_pd

# Function to perform Mean Imputation after sorting by hour
def Mean_imputation(input_data_pd):
    if 'datetime' in input_data_pd.columns:
        input_data_pd['hour'] = pd.to_datetime(input_data_pd['datetime']).dt.hour  # Extract the hour from the datetime column
        imputed_data_pd = input_data_pd.groupby('hour').apply(lambda group: group.fillna(group.mean()))
        imputed_data_pd = imputed_data_pd.reset_index(drop=True)
        imputed_data_pd = imputed_data_pd.drop(columns=['hour'])  # Drop the 'hour' column after imputation
    else:
        imputed_data_pd = input_data_pd.apply(lambda x: x.fillna(x.mean()))
    return imputed_data_pd

# Function to perform Median Imputation after sorting by hour
def Median_imputation(input_data_pd):
    if 'datetime' in input_data_pd.columns:
        input_data_pd['hour'] = pd.to_datetime(input_data_pd['datetime']).dt.hour  # Extract the hour from the datetime column
        imputed_data_pd = input_data_pd.groupby('hour').apply(lambda group: group.fillna(group.median()))
        imputed_data_pd = imputed_data_pd.reset_index(drop=True)
        imputed_data_pd = imputed_data_pd.drop(columns=['hour'])  # Drop the 'hour' column after imputation
    else:
        imputed_data_pd = input_data_pd.apply(lambda x: x.fillna(x.median()))
    return imputed_data_pd

# Function to process each file
def process_file(file_path, method):
    df = pd.read_csv(file_path)

    # Keep original PM2.5 column for later comparison
    df['PM2.5 (original)'] = df['PM2.5']

    # Drop unnecessary columns before imputation
    df_for_imputation = drop_unnecessary_columns(df).copy()

    # Apply the selected imputation method
    if method == "Interpolation":
        imputed_data_pd = Interpolation_imputation(df_for_imputation)
    elif method == "ForwardFill":
        imputed_data_pd = ForwardFill_imputation(df_for_imputation)
    elif method == "Mean":
        imputed_data_pd = Mean_imputation(df_for_imputation)
    elif method == "Median":
        imputed_data_pd = Median_imputation(df_for_imputation)

    # Get only imputed values for PM2.5
    df['PM2.5(only_imputed data)'] = imputed_data_pd['PM2.5']

    # Replace original non-missing values in 'PM2.5(only_imputed data)' with NaN
    df.loc[~df['PM2.5'].isna(), 'PM2.5(only_imputed data)'] = np.nan

    # Merge the original and imputed values
    df['full PM2.5 (after imputation)'] = df['PM2.5 (original)'].combine_first(df['PM2.5(only_imputed data)'])

    # Define output directory and create it if it doesn't exist
    output_directory = os.path.join(output_directory_base, method)
    os.makedirs(output_directory, exist_ok=True)

    # Save the final result with all necessary columns
    output_df = df[['datetime', 'missing_info', 'removed_values', 'PM2.5 (original)', 'PM2.5(only_imputed data)', 'full PM2.5 (after imputation)']]
    output_file_path = os.path.join(output_directory, os.path.basename(file_path))
    output_df.to_csv(output_file_path, index=False)

# Process all files in the input directory
for file_name in os.listdir(input_directory):
    if file_name.endswith('.csv'):
        file_path = os.path.join(input_directory, file_name)
        
        # Process with Interpolation Imputation
        process_file(file_path, method="Interpolation")
        
        # Process with Forward Fill Imputation
        process_file(file_path, method="ForwardFill")
        
        # Process with Mean Imputation
        process_file(file_path, method="Mean")
        
        # Process with Median Imputation
        process_file(file_path, method="Median")